In [ ]:
! pip3 install --quiet --upgrade google-genai google-cloud-storage

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.3/713.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 12.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.47.0 which is incompatible.


In [ ]:
from google.colab import auth
import google.auth
import google.auth.transport.requests
from google.auth.transport.requests import Request

from google.cloud import storage
from PIL import Image
import io
import requests
import json
import base64
from diffusers.utils import make_image_grid
import base64
from google import genai
from google.genai import types
from io import BytesIO
from collections import defaultdict

In [ ]:
auth.authenticate_user()
creds, PROJECT = google.auth.default()
auth_req = google.auth.transport.requests.Request()
creds.refresh(auth_req)

In [ ]:
bucket_name = "0-hongbocheng"
orignal_image_pref = "Honor_veo_template/2-person-action"

output_gcs = f"gs://{bucket_name}/{orignal_image_pref}"
gemini_project_id="veo-testing"
# gemini_project_id="honor-cloud-ai-testteam"
veo_project_id = "veo-testing"
# veo_project_id = "honor-cloud-ai-testteam"
location="us-central1"
model_name = "veo-3.1-fast-generate-preview"
use_gemini_based_prompt = True
override_video = True
only_generate_prompt = False
generated_sample_count = 1

video_configurations = [{"suffix":"_2_person_hug_grassland","video_background": "On the grassland", "action":"They stood face to face and embraced."}, {"suffix":"_2_person_highfive_court","video_background": "an empty basketball court", "action":"They high-fived each other face to face, and hugged each other."}]

In [ ]:
def generate_veo_promt (project_id,location, image_url, image_url_2, video_configuration):
  client = genai.Client(
      vertexai=True,
      project=gemini_project_id,
      location="global",
  )
  system_instruction="""
# Role: Expert Dual-Subject Video Prompt Engineer & Safety Validator (Veo Specialist)

**Specialization:** Multi-Character Interaction, Character-Centric Composition, Strict Safety Compliance, and High-Fidelity Portraiture.

**Objective:**
Your primary goal is to synthesize two reference images into a single, cohesive video prompt for **Google Veo**.
You must first **VALIDATE** the inputs for safety and quality. If validated, generate a prompt that prioritizes **close-range interaction**, ensuring characters remain large, clear, and the focal point of the video.

---

## **YOUR INPUTS:**

1.  **Image A:** Visual reference for the first person.
2.  **Image B:** Visual reference for the second person.
3.  **Action:** The specific interaction between Person A and Person B.
4.  **Background:** The environment where the video takes place.

---

## **YOUR ANALYSIS PROCESS:**

### **Phase 1: Mandatory Validation Protocol (GATEKEEPER)**
Before generating any prompt, you must perform these three checks. **If ANY check fails, set JSON `status` to "failed" and stop.**

1.  **Subject Count & Type Check:**
    * Confirm that **exactly two distinct human subjects** are identified (one per image).
    * *Reject if:* Only one person is found, or if subjects are animals/objects.
2.  **Fidelity & Suitability Check:**
    * Assess facial clarity.
    * *Reject if:* Faces are pixelated, blurry, or too far away in the original photos to support detailed animation.
3.  **Safety & Content Moderation (STRICT):**
    * Scan for NSFW (Not Safe For Work) content.
    * *Reject if:* The inputs or requested `Action` imply nudity, sexual acts, excessive violence, or pornography.
    * *Note:* Romantic gestures (hugging, holding hands) are allowed; explicit content is forbidden.

### **Phase 2: Visual Synthesis & Interaction Design**
* **Character Integration:** Extract details (hair, clothing, ethnicity) from Image A and Image B.
* **Interaction Logic:** Integrate the `Action` parameter. Ensure the physical interaction is spatially logical.
* **Environment Setting:** Apply the `Background`. Ensure background lighting matches the subjects' position.

### **Phase 3: Cinematic Polish & Framing Control (CRITICAL)**
* **Proximity Constraint (Subject Dominance):**
    * **DO NOT** use "Wide Shot," "Extreme Long Shot," or "Bird's Eye View."
    * **MUST USE** framing that keeps characters large and visible:
        * *"Medium Shot" (Waist up)* - Best for most interactions.
        * *"Two-Shot" (Both faces clearly visible)*.
        * *"Over-the-shoulder"* - For conversation.
        * *"Cowboy Shot" (Thighs up)* - For walking/dancing actions.
* **Ratio Guideline:** Subjects should occupy **at least 50-60%** of the vertical frame height. Avoid placing them deep in the background.
* **Camera Movement:** Use subtle movements that track the characters closely (e.g., "Slow tracking shot following the characters," "Subtle push-in on their expressions").

---

## **OUTPUT FORMAT:**

Return the result **STRICTLY** in the following JSON format.

```json
{
  "status": "success | failed",
  "reason": "If failed, strictly explain why (e.g., 'NSFW content detected', 'Image B too blurry'). If success, leave null.",
  "veo_prompt": "The final generated prompt string. (Leave empty if status is failed)"
}
"""

  image1 = types.Part.from_uri(
      file_uri=image_url,
      mime_type="image/jpeg",
  )
  image2 = types.Part.from_uri(
      file_uri=image_url_2,
      mime_type="image/jpeg",
  )

  model = "gemini-2.5-flash-lite"
  prompt = f"""the video background is: {video_configuration["video_background"]}, action is:{video_configuration["action"]}"""
  contents = [
    types.Content(
      role="user",
      parts=[
        image1, image2,
        types.Part.from_text(text=prompt)
      ]
    )
  ]

  generate_content_config = types.GenerateContentConfig(
    temperature = 1,
    top_p = 0.95,
    max_output_tokens = 65535,
    safety_settings = [types.SafetySetting(
      category="HARM_CATEGORY_HATE_SPEECH",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_DANGEROUS_CONTENT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_HARASSMENT",
      threshold="OFF"
    )],
    response_mime_type = "application/json",
    thinking_config=types.ThinkingConfig(
      thinking_budget=512,
    ),
    system_instruction=[system_instruction],
  )

  responses = client.models.generate_content(
    model = model,
    contents = contents,
    config = generate_content_config,
    )
  response_results = [{"status": "failed"}]
  try:
    response_results = json.loads(responses.text)
  except Exception as e:
          print(e)
  print (f"veo_prompt={response_results}")
  return response_results

In [ ]:
def get_operation_status(PROJECT_NAME,model_name,operation_name):
  request = Request()
  creds.refresh(request=request)
  access_token = creds.token
  headers = {
      'Authorization': f'Bearer {access_token}',
      'Content-Type': 'application/json;charset=utf-8',
  }

  # url = f"""https://us-central1-autopush-aiplatform.sandbox.googleapis.com/v1beta1/projects/{PROJECT_NAME}/locations/us-central1/publishers/google/models/{model_name}:fetchPredictOperation"""
  url = f"https://{location}-aiplatform.googleapis.com/v1/projects/{veo_project_id}/locations/{location}/publishers/google/models/{model_name}:fetchPredictOperation"
  params = {"operationName":operation_name}
  response = requests.post(url,headers=headers,json=params)
  return response


  #Image--->Gemini---->VEO_PROMPT + IMAGE (ID) ----> VEO Model-----> (operation_name, image ID) (DB/Redis) ----> future call-call operation_status (operation_name) ---->

In [ ]:
def get_image_prompt_params(veo_prompt,OUTPUT_GCS_URI ,Ratio, img1_uri, img1_base64,img2_uri,img2_base64):
  if img1_base64 and img2_base64:
    #print(f" use image_base64")
    params ={
        "instances": [
          {
            "prompt": veo_prompt,
            "referenceImages": [{
                "image": {
                  "bytesBase64Encoded": img1_base64,
                  "mimeType": "image/jpg"
                },
                "referenceType": "asset"
            },{
                "image": {
                  "bytesBase64Encoded": img2_base64,
                  "mimeType": "image/jpg"
                },
                "referenceType": "asset"
            }]
          }
        ],
        "parameters": {
          "storageUri": OUTPUT_GCS_URI,
          "aspectRatio": Ratio,
          "sampleCount": generated_sample_count,
          "durationSeconds": 8,
          "personGeneration": "allow_all",
          #"enhancePrompt": True
        }
    }

    return params
  else:
    params ={
        "instances": [
          {
            "prompt": veo_prompt,
            "referenceImages": [{
                "image": {
                  "gcsUri": img1_uri,
                  "mimeType": "image/jpg"
                },
                "referenceType": "asset"
            },{
                "image": {
                  "gcsUri": img2_uri,
                  "mimeType": "image/jpg"
                },
                "referenceType": "asset"
            }]
          }
        ],
        "parameters": {
          "storageUri": OUTPUT_GCS_URI,
          "aspectRatio": Ratio,
          "sampleCount": generated_sample_count,
          "durationSeconds": 8,
          "personGeneration": "allow_all",
          "enhancePrompt": True
        }
    }

    return params

## generate video by image
def generate_video_by_image_with_prompt(PROJECT_NAME, model_name, input_prompt,OUTPUT_GCS_URI,img1_uri = None, img1_base64= None, img2_uri=None,img2_base64=None,Ratio = "9:16"):
    print("generate_video_by_image_with_prompt")
    try:
      # Generate access token (with debugging)
      request = Request()
      creds.refresh(request=request)

      # print("Token Request Status Code:", request.response.status_code)  # Check status
      # print("Token Request Response:", request.response.text)  # Check response

      access_token = creds.token


      # print("Access Token:", access_token)  # Print the token
      headers = {
          'Authorization': f'Bearer {access_token}',
          'Content-Type': 'application/json;charset=utf-8',
      }

      # url = f"""https://us-central1-autopush-aiplatform.googleapis.sandbox.com/v1beta1/projects/{PROJECT_NAME}/locations/us-central1/publishers/google/models/{model_name}:predictLongRunning"""
      # url = f"""https://us-central1-autopush-aiplatform.sandbox.googleapis.com/v1beta1/projects/{PROJECT_NAME}/locations/us-central1/publishers/google/models/veo-2.0-generate-001:predictLongRunning"""
      url = f"https://{location}-aiplatform.googleapis.com/v1/projects/{veo_project_id}/locations/{location}/publishers/google/models/{model_name}:predictLongRunning"
      params = get_image_prompt_params(input_prompt,OUTPUT_GCS_URI,Ratio = Ratio, img1_uri = img1_uri, img1_base64=img1_base64,img2_uri=img2_uri,img2_base64=img2_base64)
      print(f" params={params} url={url}")
      response = requests.post(url,headers=headers,json=params)
      #print(f" response={response}")
      #print(response.text)
    except Exception as e:
      print(e)
    return response


In [ ]:
def copy_gcs_file(source_blob_uri, destination_blob_uri):
    """
    Copies a single GCS file from one location to another.

    Args:
        source_blob_uri (str): The URI of the source blob (e.g., "gs://bucket-name/source/file.mp4").
        destination_blob_uri (str): The URI of the destination blob (e.g., "gs://other-bucket/destination/file.mp4").
    """

    storage_client = storage.Client()

    # Parse source URI
    source_parts = source_blob_uri.replace("gs://", "").split("/")
    source_bucket_name = source_parts[0]
    source_blob_name = "/".join(source_parts[1:])

    # Parse destination URI
    destination_parts = destination_blob_uri.replace("gs://", "").split("/")
    destination_bucket_name = destination_parts[0]
    destination_blob_name = "/".join(destination_parts[1:])

    source_bucket = storage_client.bucket(source_bucket_name)
    source_blob = source_bucket.blob(source_blob_name)

    destination_bucket = storage_client.bucket(destination_bucket_name)

    # Copy the blob
    new_blob = source_bucket.copy_blob(
        source_blob, destination_bucket, destination_blob_name
    )

    #print(f" Copied {source_blob_uri} to {destination_blob_uri}")

def list_blobs(bucket_name, prefix):
    """Lists all the blobs in the bucket that begin with the prefix.

    Args:
        bucket_name: The name of the GCS bucket.
        prefix: The prefix of the GCS blobs to list.

    Returns:
        A list of blob names (strings).
    """

    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    blobs = bucket.list_blobs(prefix=prefix) #Get the blobs matching the prefix
    results = []
    for blob in blobs:
        if not blob.name.endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp')):
            continue  # 忽略非图像文件
        current_name = f"gs://{bucket_name}/{blob.name}"
        mp4_blob = bucket.blob(blob.name.rsplit('.', 1)[0] + '.mp4')
        if not override_video and mp4_blob.exists():
          continue # 如果已经生成过，直接忽略.
        try:
            blob_content = blob.download_as_bytes()
            image = Image.open(io.BytesIO(blob_content))
            width, height = image.size
            if width < height * 0.9:
                results.append({"name": blob.name, "ratio":"9:16", "min": width})
            else:
                results.append({"name": blob.name, "ratio":"16:9", "min": height})

        except Exception as e:
            print(f" 处理 {blob.name} 时出错：{e}")

    return results

images_with_ratio = list_blobs(bucket_name, orignal_image_pref)
print(images_with_ratio)

def group_data (images_with_ratio):
  # 用于分组的字典
  grouped_data = defaultdict(list)

  for img in images_with_ratio:
      # 提取文件名，例如 "1-1.jpg"
      filename = img['name'].split('/')[-1]
      # 提取组ID，即 "-" 前面的数字
      group_id = filename.split('-')[0]
      grouped_data[group_id].append(img)

  # 转换为目标 JSON 格式
  result = []
  for group_id, images in grouped_data.items():
      result.append({
          "group_id": group_id,
          "images": images
      })
  return result

group_images_with_ratio = group_data(images_with_ratio)
print(f"group_images_with_ratio={group_images_with_ratio}")

[{'name': 'Honor_veo_template/2-person-action/1-1.jpg', 'ratio': '9:16', 'min': 1080}, {'name': 'Honor_veo_template/2-person-action/1-2.jpg', 'ratio': '9:16', 'min': 1080}, {'name': 'Honor_veo_template/2-person-action/2-1.jpg', 'ratio': '16:9', 'min': 2592}, {'name': 'Honor_veo_template/2-person-action/2-2.jpg', 'ratio': '9:16', 'min': 1080}, {'name': 'Honor_veo_template/2-person-action/3-1.jpg', 'ratio': '9:16', 'min': 1080}, {'name': 'Honor_veo_template/2-person-action/3-2.jpg', 'ratio': '9:16', 'min': 3072}]
group_images_with_ratio=[{'group_id': '1', 'images': [{'name': 'Honor_veo_template/2-person-action/1-1.jpg', 'ratio': '9:16', 'min': 1080}, {'name': 'Honor_veo_template/2-person-action/1-2.jpg', 'ratio': '9:16', 'min': 1080}]}, {'group_id': '2', 'images': [{'name': 'Honor_veo_template/2-person-action/2-1.jpg', 'ratio': '16:9', 'min': 2592}, {'name': 'Honor_veo_template/2-person-action/2-2.jpg', 'ratio': '9:16', 'min': 1080}]}, {'group_id': '3', 'images': [{'name': 'Honor_veo_tem

In [ ]:
upscale_url=f"https://{location}-aiplatform.googleapis.com/v1/projects/{gemini_project_id}/locations/{location}/publishers/google/models/imagegeneration@002:predict"

def build_upscale_params(gcs_uri, upscale_factor):
  #print(f" upscale_factor={upscale_factor} gcs_uri={gcs_uri}")
  return {
    "instances": [
      {
        "prompt": "",
        "image": {
          "gcsUri": gcs_uri
        },
      }
    ],
    "parameters": {
      "sampleCount": 1,
      "mode": "upscale",
      "upscaleConfig": {
        "upscaleFactor": upscale_factor
      }
    }
  }

def upscale_image(gcs_uri, upscale_factor):
  # Generate access token (with debugging)
    request = Request()
    creds.refresh(request=request)

    # print("Token Request Status Code:", request.response.status_code)  # Check status
    # print("Token Request Response:", request.response.text)  # Check response
    access_token = creds.token
    # print("Access Token:", access_token)  # Print the token
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json;charset=utf-8',
    }

    params = build_upscale_params(gcs_uri, upscale_factor)

    response = requests.post(upscale_url,headers=headers,json=params)
    response_json = json.loads(response.text)
    print(response_json['predictions'][0]['mimeType'])
    return response_json['predictions'][0]['bytesBase64Encoded']

#upscale_image(f"gs://{bucket_name}/veo/issues/stop-9.jpg", "x2")


In [ ]:
from google.cloud import storage

def write_request_to_gcs(bucket_name, destination_blob_name, text_content):
    """Writes text content to a file in Google Cloud Storage."""
    # Instantiates a client
    storage_client = storage.Client() # Assumes ADC or GOOGLE_APPLICATION_CREDENTIALS is set

    # Get the bucket
    bucket = storage_client.bucket(bucket_name)

    # Get the blob (file object)
    blob = bucket.blob(destination_blob_name)

    # Upload the text string
    blob.upload_from_string(text_content)

    #print(f" Text successfully written to gs://{bucket_name}/{destination_blob_name}")

In [ ]:
# images_with_ratio = [{'name': 'veo/0508_5/老照片 (7).jpg', 'ratio': '9:16', 'min': 1200}]

In [ ]:
import concurrent.futures
import json
import time
import threading # 用于获取当前线程信息

# --- 重要假设 ---
# 确认以下辅助函数是 *同步* 的阻塞版本:
# def generate_veo_promt(project_id, location, image_url): ...
# def write_request_to_gcs(bucket, blob_name, content): ...
# def upscale_image(image_url, scale): ... # 返回 base64 字符串
# def generate_video_by_image_with_prompt(project_id, model, prompt, output_gcs, image_url, base64_image, ratio): ... # 返回 requests.Response 对象或类似对象
# def get_operation_status(project_id, model, operation_name): ... # 返回 requests.Response 对象或类似对象
# def copy_gcs_file(source_gcs_uri, destination_gcs_uri): ...
# --- 结束假设 ---

# --- 线程安全注意事项 ---
# - 如果你的辅助函数依赖于非线程安全的全局状态或共享对象，你需要添加锁或其他同步机制。
# - 对于 Google Cloud 客户端库：通常，为每个请求创建一个新的客户端实例是安全的，
#   或者如果客户端对象本身被设计为线程安全的，则可以共享一个实例。
#   为简单起见，此示例假设辅助函数内部正确处理了客户端的线程安全。
# ---

def process_single_image_thread(group_image_with_ratio, video_configuration, bucket_name, orignal_image_pref, use_gemini_based_prompt, only_generate_prompt, gemini_project_id, location, veo_project_id, model_name, output_gcs):
    """
    在单个线程中处理单个图片的同步函数。
    """
    image_name = group_image_with_ratio['images'][0].get('name', 'UNKNOWN_IMAGE')
    image_name_2 = group_image_with_ratio['images'][1].get('name', 'UNKNOWN_IMAGE')
    print(f"images are image_name={image_name} image_name_2={image_name_2}")
    thread_name = threading.current_thread().name # 获取当前线程名，方便日志追踪
    try:
        #print(f" [{thread_name}] 开始处理: {image_name}")
        orignal_image_url = f"gs://{bucket_name}/{image_name}"
        orignal_image_url_2 = f"gs://{bucket_name}/{image_name_2}"
        veo_prompt_result = ""

        if use_gemini_based_prompt:
            # print(f" [{thread_name}][{image_name}] 使用 Gemini 生成提示...")
            # *** 调用同步版本的辅助函数 ***
            veo_prompt_result = generate_veo_promt(gemini_project_id, location, orignal_image_url, orignal_image_url_2, video_configuration)

            if veo_prompt_result['status'] == "failed":
              result = f"""{image_name} checked failed: {veo_prompt_result['reason']}"""
              print(result)
              return result

            veo_prompt = veo_prompt_result['veo_prompt']
            #

            # veo_prompt = "the girl is dancing after meal."
            prompt_blob_name = image_name.rsplit('.', 1)[0] + video_configuration['suffix'] + '.txt'
            print(f" [{thread_name}][{image_name}] 将提示写入 gs://{bucket_name}/{prompt_blob_name}")
            # *** 调用同步版本的辅助函数 ***
            write_request_to_gcs(bucket_name, prompt_blob_name, veo_prompt)

            if only_generate_prompt:
                #print(f" [{thread_name}][{image_name}] 仅生成提示完成。")
                return f"{image_name}: Prompt generated" # 返回状态信息

        if only_generate_prompt: # 再次检查
             return f"{image_name}: Prompt only mode"

        base64_image = None
        # if min_value < 720:
        #     #print(f" [{thread_name}][{image_name}] 放大图片 (min_value={min_value})...")
        #     # *** 调用同步版本的辅助函数 ***
        #     base64_image = upscale_image(orignal_image_url, "x2")
        #     #print(f" [{thread_name}][{image_name}] 图片放大完成。")

        print(f" [{thread_name}][{image_name}][{image_name_2}] 提交视频生成请求...")
        # *** 调用同步版本的辅助函数 ***
        # 假设返回一个类似 requests.Response 的对象，具有 .text 属性
        aspect_ratio = '16:9'#image_with_ratio['ratio']
        #PROJECT_NAME, model_name, input_prompt,OUTPUT_GCS_URI,img1_uri = None, img1_base64= None, img2_uri=None,img2_base64=None,Ratio = "9:16"
        operation_response = generate_video_by_image_with_prompt(
            veo_project_id, model_name, veo_prompt, output_gcs,
            orignal_image_url, None, orignal_image_url_2, base64_image, aspect_ratio
        )
        # 检查响应状态码可能是个好主意 (例如 operation_response.raise_for_status())
        operation_data = json.loads(operation_response.text) # 解析响应文本
        operation_name = operation_data['name']
        #print(f" [{thread_name}][{image_name}] 请求已提交. 操作名称: {operation_name}")

        # 轮询循环 - 在线程中 time.sleep 是可以接受的
        while True:
            #print(f" [{thread_name}][{image_name}] 检查操作状态: {operation_name}...")
            # *** 调用同步版本的辅助函数 ***
            operation_status_response = get_operation_status(veo_project_id, model_name, operation_name)
            # 同样，检查响应状态码
            status_text = operation_status_response.text # 获取响应文本
            #print(status_text)

            try:
                response_json = json.loads(status_text)
                # 检查完成状态 (根据实际 API 响应调整)
                if response_json.get("done") is True:
                    #print(f" [{thread_name}][{image_name}] 操作 {operation_name} 完成。")
                    break # 退出轮询

                # 检查错误状态 (根据实际 API 响应调整)
                if "error" in response_json and response_json["error"] is not None:
                     print(f" [{thread_name}][{image_name}] 操作 {operation_name} 失败: {response_json['error']}")
                     # 抛出异常，让 Future 捕获它
                     raise RuntimeError(f"Operation {operation_name} failed: {response_json['error']}")

                #print(f" [{thread_name}][{image_name}] 操作 {operation_name} 仍在进行中...")

            except json.JSONDecodeError:
                print(f" [{thread_name}][{image_name}] 警告: 无法解析状态响应JSON: {status_text[:200]}...")
            except Exception as poll_err:
                print(f" [{thread_name}][{image_name}] 解析操作 {operation_name} 状态时出错: {poll_err}. 响应: {status_text[:200]}...")
                # 决定是否应该因为这个错误而终止

            time.sleep(5) # 同步休眠
        #print(f"response_json={response_json}")
        # 处理成功完成的操作结果
        final_videos = response_json.get('response', {}).get('videos', [])
        if not final_videos:
             print(f" [{thread_name}][{image_name}] 警告: 操作完成但响应中未找到视频 {operation_name}. {response_json}")

        for i, response_in_status in enumerate(final_videos):
            video_orignal_url = response_in_status.get('gcsUri')
            if not video_orignal_url:
                print(f" [{thread_name}][{image_name}] 警告: 视频条目 {i} 没有 gcsUri.")
                continue

            #print(f" [{thread_name}][{image_name}] 结果视频 GCS URL: {video_orignal_url}")
            post_fix = f"_{i}" if i > 0 else ""
            # 构造目标 GCS 路径
            base_image_filename = image_name.rsplit('/', 1)[-1].rsplit('.', 1)[0]
            target_blob_name = f"{orignal_image_pref}/{base_image_filename}{post_fix}{video_configuration['suffix']}.mp4"
            #print(f" target_blob_name={target_blob_name}")
            target_gcs_uri = f"gs://{bucket_name}/{target_blob_name}"

            #print(f" [{thread_name}][{image_name}] 正在复制 {video_orignal_url} 到 {target_gcs_uri}...")
            # *** 调用同步版本的辅助函数 ***
            copy_gcs_file(video_orignal_url, target_gcs_uri)
            #print(f" [{thread_name}][{image_name}] 视频 {i} 复制完成.")

        #print(f" [{thread_name}][{image_name}] 处理完成。")
        return f"{image_name}: Processing successful" # 返回成功信息

    except Exception as e:
        # 记录特定于此图像的错误
        print(f" [{thread_name}] 处理图像 '{image_name}' 时发生错误: {e}")
        # 重新抛出异常，以便 concurrent.futures 可以捕获它
        raise

def run_multithreaded_image_processing(group_images_with_ratio, bucket_name, orignal_image_pref, use_gemini_based_prompt, only_generate_prompt, gemini_project_id, location, veo_project_id, model_name, output_gcs, max_workers=5):
    """
    使用 ThreadPoolExecutor 创建并运行所有图像处理任务。

    Args:
        max_workers (int): 同时运行的最大线程数。调整此值以控制并发度，
                         避免耗尽资源或达到 API 速率限制。
    """
    if not group_images_with_ratio:
        #print(f" 在 {bucket_name}/{orignal_image_pref} 中没有找到需要处理的图片。")
        return

    #print(f" 找到 {len(images_with_ratio)} 张图片。使用最多 {max_workers} 个线程开始并发处理...")

    futures_map = {} # 用于将 future 映射回 image_name 或 info
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        for image_info in group_images_with_ratio:
            # 提交任务到线程池
            #user_input = "带墨镜，穿豹纹睡衣坐在椅子上打电话,说'I dont want to eat noodles, give me bones.'"
            for video_configuration in video_configurations:
              future = executor.submit(
                  process_single_image_thread, # 要执行的函数
                  # 函数所需的参数:
                  image_info,
                  video_configuration,
                  bucket_name,
                  orignal_image_pref,
                  use_gemini_based_prompt,
                  only_generate_prompt,
                  gemini_project_id,
                  location,
                  veo_project_id,
                  model_name,
                  output_gcs
              )
              print(image_info['images'])
              futures_map[future] = image_info['images'][0].get('name', 'UNKNOWN_IMAGE') # 存储 future 和图像名称的映射

        print("所有任务已提交。等待任务完成...")
        # 'with' 块结束时，会自动等待所有提交的任务完成 (executor.shutdown(wait=True))

    # 任务完成后，检查结果和异常
    print("\n--- 多线程处理摘要 ---")
    success_count = 0
    failure_count = 0
    # 使用 as_completed 可以按完成顺序获取结果，但这里我们按提交顺序检查
    for future, image_name in futures_map.items():
        try:
            # 调用 result() 会获取线程的返回值，
            # 如果线程中发生未捕获的异常，result() 会重新引发该异常
            result = future.result()
            #print(f" 图片 {image_name}: 成功完成。返回: {result}")
            success_count += 1
        except Exception as e:
            # 捕获在线程中引发并在 result() 中重新引发的异常
            print(f" 图片 {image_name}: 失败 - {type(e).__name__}: {e}")
            failure_count += 1

    print(f" 总计: {len(images_with_ratio)}, 成功: {success_count}, 失败: {failure_count}")
    print("-----------------------")


In [ ]:

worker_count = 1

try:
      run_multithreaded_image_processing(
          group_images_with_ratio, bucket_name, orignal_image_pref,
          use_gemini_based_prompt, only_generate_prompt,
          gemini_project_id, location, veo_project_id, model_name, output_gcs,
          max_workers=4
      )
except Exception as e:
      print(f" 在多线程处理的主流程中发生错误: {e}")

images are image_name=Honor_veo_template/2-person-action/1-1.jpg image_name_2=Honor_veo_template/2-person-action/1-2.jpg
[{'name': 'Honor_veo_template/2-person-action/1-1.jpg', 'ratio': '9:16', 'min': 1080}, {'name': 'Honor_veo_template/2-person-action/1-2.jpg', 'ratio': '9:16', 'min': 1080}]
images are image_name=Honor_veo_template/2-person-action/1-1.jpg image_name_2=Honor_veo_template/2-person-action/1-2.jpg
[{'name': 'Honor_veo_template/2-person-action/1-1.jpg', 'ratio': '9:16', 'min': 1080}, {'name': 'Honor_veo_template/2-person-action/1-2.jpg', 'ratio': '9:16', 'min': 1080}]
images are image_name=Honor_veo_template/2-person-action/2-1.jpg image_name_2=Honor_veo_template/2-person-action/2-2.jpg
[{'name': 'Honor_veo_template/2-person-action/2-1.jpg', 'ratio': '16:9', 'min': 2592}, {'name': 'Honor_veo_template/2-person-action/2-2.jpg', 'ratio': '9:16', 'min': 1080}]
images are image_name=Honor_veo_template/2-person-action/2-1.jpg image_name_2=Honor_veo_template/2-person-action/2-2.j